# Tutorial 2: **Notebook** - *FVF*

> 🚧 **Under Construction**

**By 
gLayout Team**

__Content creators:__ Subham Pal, Saptarshi Ghosh

__Content reviewers:__ Mehedi Saligne

___
# Tutorial Objectives

This notebook is a tutorial on-

- **LVS (Layout Versus Schematic):**  
  You will learn how to compare your physical layout with the original schematic to ensure they are functionally identical. This process helps catch connectivity or device mismatches before fabrication.

- **Extraction and Simulation:**  
  The tutorial will guide you through extracting parasitic elements from your layout, such as capacitance and resistance, to create a more accurate circuit model. You will then simulate the extracted netlist to analyze and verify the real-world performance of your design.

## **Target** **Block** : **Inverter Cell**

The CMOS inverter is a fundamental digital circuit building block, widely used due to its low static power consumption, high noise immunity, and fast switching characteristics. It consists of a complementary pair of MOSFETs: one PMOS and one NMOS transistor, with their gates and drains connected together. The source of the PMOS is connected to the supply voltage (VDD), while the source of the NMOS is connected to ground (VSS).

### **Operation Principle**
- **Input Low (Logic 0):** The PMOS transistor turns ON and the NMOS turns OFF. This connects the output (VOUT) to VDD, producing a logic high (1) at the output.
- **Input High (Logic 1):** The NMOS transistor turns ON and the PMOS turns OFF. This connects the output to ground, producing a logic low (0) at the output.

This complementary switching action ensures that the output is always the logical inverse of the input, making the CMOS inverter a true logic inverter.

![](_images/CMOS_Inverter.png)

```bibtex
Wikimedia Commons contributors. (2010). CMOS Inverter Schematic [Diagram]. Wikimedia Commons. https://commons.wikimedia.org/wiki/File:CMOS_Inverter.svg
```


## **NetList generation and LVS**
let's go through the step by step procedure to generate LVS and DRC clean layout of a FVF cell.

In [ ]:
# Environment bootstrap — works in both iic-osic-tools and bare-venv setups.
# iic-osic-tools defines PDK_ROOT / PDK / PDKPATH in ~/.bashrc but jupyter
# kernels launched from its menu don't inherit them, so we re-source bashrc.
# setdefault() means we never overwrite variables already exported by the
# caller. In bare-venv setups, PDKPATH is derived from PDK_ROOT/PDK if not
# already set. See tutorial/HOW_TO_RUN.md for the env-var contract.
import os
import subprocess
try:
    _printenv = subprocess.run(
        ['bash', '-c', 'source ~/.bashrc 2>/dev/null && printenv'],
        text=True, capture_output=True, timeout=10,
    ).stdout
    for _line in _printenv.splitlines():
        if '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k, _v)
except Exception:
    pass
if 'PDK_ROOT' in os.environ and 'PDK' in os.environ:
    os.environ.setdefault('PDKPATH', os.path.join(os.environ['PDK_ROOT'], os.environ['PDK']))


In [ ]:
from glayout import MappedPDK, sky130 , gf180
from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle

import gdsfactory as gf
gf.clear_cache()

In [ ]:
from glayout import nmos, pmos
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring

In [ ]:
from glayout.util.comp_utils import evaluate_bbox, prec_center, prec_ref_center, align_comp_to_port
from glayout.util.port_utils import add_ports_perimeter,print_ports
from glayout.util.snap_to_grid import component_snap_to_grid
from glayout.spice.netlist import Netlist

In [ ]:
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route

FVF has two fets as shown in the schematic. We call M1 as input fet and M2 as feedback fet. Lets define arguments for the FETs

### 2. Basic Usage of the GLayout Framework
Each generator is a Python function that takes a `MappedPDK` object as a parameter and generates a DRC clean layout for the given PDK. The generator may also accept a set of optional layout parameters such as the width or length of a MOSFET. All parameters are normal Python function arguments.

The generator returns a `GDSFactory.Component` object that can be written to a `.gds` file and viewed using a tool such as Klayout. In this example, the `gdstk` library is used to convert the `.gds` file to an SVG image for viewing.

The pre-PEX SPICE netlist for the component can be viewed using `component.info['netlist'].generate_netlist()`.

In the following example the FET generator `glayout.primitives.fet` is imported and run with both the [Skywater 130](https://skywater-pdk.readthedocs.io/en/main/) and [GF180](https://gf180mcu-pdk.readthedocs.io/en/latest/) PDKs.

#### Demonstration of Basic Layout / Netlist Generation in SKY130 & GF180

In [ ]:
# from glayout import nmos,sky130,gf180

# import gdstk
# import svgutils.transform as sg
# import IPython.display
# from IPython.display import clear_output
# import ipywidgets as widgets

# # Used to display the results in a grid (notebook only)
# left = widgets.Output()
# leftSPICE = widgets.Output()
# right = widgets.Output()
# rightSPICE = widgets.Output()
# hide = widgets.Output()

# grid = widgets.GridspecLayout(1, 4)
# grid[0, 0] = left
# grid[0, 1] = leftSPICE
# grid[0, 2] = right
# grid[0, 3] = rightSPICE
# display(grid)

# def display_gds(gds_file, scale = 3):
#   # Generate an SVG image
#   top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
#   top_level_cell.write_svg('out.svg')

#   # Scale the image for displaying
#   fig = sg.fromfile('out.svg')
#   fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
#   fig.save('out.svg')

#   # Display the image
#   IPython.display.display(IPython.display.SVG('out.svg'))

# def display_component(component, scale = 3):
#   # Save to a GDS file
#   with hide:
#     component.write_gds("out.gds")

#   display_gds('out.gds', scale)

# with hide:
#   # Generate the sky130 component
#   component_sky130 = nmos(pdk = sky130, fingers=5)
#   # Generate the gf180 component
#   component_gf180 = nmos(pdk = gf180, fingers=5)

# # Display the components' GDS and SPICE netlists
# with left:
#   print('Skywater 130nm N-MOSFET (fingers = 5)')
#   display_component(component_sky130, scale=2)
# with leftSPICE:
#   print('Skywater 130nm SPICE Netlist')
#   print(component_sky130.info['netlist'].generate_netlist())

# with right:
#   print('GF 180nm N-MOSFET (fingers = 5)')
#   display_component(component_gf180, scale=2)
# with rightSPICE:
#   print('GF 180nm SPICE Netlist')
#   print(component_gf180.info['netlist'].generate_netlist())

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("./INV"))

from my_INV import inverter,add_inv_labels

In [ ]:
comp = inverter(gf180)
for s in comp.references:
    print(s.name)  # This will show 'fet_P', 'fet_N', etc.

In [ ]:
import tempfile
from pathlib import Path

magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"

design_name = comp.name
path_to_dir = Path("./INV").resolve() / "ext" / design_name
path_to_dir.mkdir(parents=True, exist_ok=True)

pex_path = path_to_dir / f"{design_name}_pex.spice"
gds_path = path_to_dir / f"{design_name}.gds"
comp.write_gds(str(gds_path))

magic_script = f"""
drc off
gds flatglob *\\$\\$*
gds read {gds_path}
flatten {design_name}
load {design_name}
select top cell
extract do local
extract all
ext2sim labels on
ext2sim
extresist tolerance 10
extresist
ext2spice lvs
ext2spice cthresh 0
ext2spice extresist on
ext2spice -o {pex_path}
exit
"""
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.tcl') as f:
    f.write(magic_script)
    magic_script_path = f.name

magic_cmd = f"magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}"
res = subprocess.run(magic_cmd, shell=True, capture_output=True, text=True)
print(res.stdout[-800:])
if res.returncode != 0:
    print("STDERR:", res.stderr[-800:])
print("Wrote:", pex_path if pex_path.exists() else '(extraction did not produce a .spice file)')


In [ ]:
comp = add_inv_labels(comp, gf180)
comp.name = "INV"
#comp.write_gds('out_INV.gds')
comp.show()
#print("...Running DRC...")
drc_result = gf180.drc_magic(comp, "INV")

In [ ]:
# Build a SPICE netlist for the inverter using the fet references that
# my_INV.inverter() stashed on top_level.info (the same pattern FVF uses).
def inv_netlist(inv_in: Component) -> Component:
    fet_p = inv_in.info["fet_P"]
    fet_n = inv_in.info["fet_N"]
    netlist = Netlist(circuit_name='INVERTER', nodes=['VIN', 'VBULK', 'VOUT', 'VDD'])
    netlist.connect_netlist(fet_p.info['netlist'],
                            [('D', 'VOUT'), ('G', 'VIN'), ('S', 'VDD'), ('B', 'VBULK')])
    netlist.connect_netlist(fet_n.info['netlist'],
                            [('D', 'VOUT'), ('G', 'VIN'), ('S', 'VBULK'), ('B', 'VBULK')])
    inv_in.info['netlist'] = netlist
    return inv_in

comp = inv_netlist(comp)
print(comp.info['netlist'].generate_netlist())


In [ ]:
print('Inverter component:', comp.name)
print('ports:', list(comp.ports)[:8], '...')


### Run LVS
Design Rule Check ensures that the physical layout of an integrated circuit adheres to the manufacturing constraints defined by the foundry, such as minimum spacing, width, and enclosure rules. `Magic` is the tool we use for DRC here.

In [ ]:
# Run netgen LVS against the labeled inverter layout.
netgen_lvs_result = gf180.lvs_netgen(comp, comp.name)


## Extraction and Post-Pex Simulation

In [ ]:
import tempfile
from pathlib import Path

magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"

design_name = comp.name
path_to_dir = Path("./INV").resolve() / "ext" / design_name
path_to_dir.mkdir(parents=True, exist_ok=True)

pex_path = path_to_dir / f"{design_name}_pex.spice"
gds_path = path_to_dir / f"{design_name}.gds"
comp.write_gds(str(gds_path))

magic_script = f"""
drc off
gds flatglob *\\$\\$*
gds read {gds_path}
flatten {design_name}
load {design_name}
select top cell
extract do local
extract all
ext2sim labels on
ext2sim
extresist tolerance 10
extresist
ext2spice lvs
ext2spice cthresh 0
ext2spice extresist on
ext2spice -o {pex_path}
exit
"""
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.tcl') as f:
    f.write(magic_script)
    magic_script_path = f.name

magic_cmd = f"magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}"
res = subprocess.run(magic_cmd, shell=True, capture_output=True, text=True)
print(res.stdout[-800:])
if res.returncode != 0:
    print("STDERR:", res.stderr[-800:])
print("Wrote:", pex_path if pex_path.exists() else '(extraction did not produce a .spice file)')
